In [1]:
import pandas as pd
import numpy as np
import os
from typing import List, Tuple
from dataset_utils import split_data, TimeSeriesDataset, min_max_normalization
from model.lstm import LSTMMdel
from model.nlinear import NLinear
from model.SegRNN import SegRNN
from model.PatchTST import PatchTST,Config
from train_utils import Trainer
from evaluate_utils import Evaluator
import torch
import torch.nn as nn
import torch.nn.init as init
from torch.utils.data import DataLoader
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from collections import defaultdict
from typing import Dict
import threading
import json
from json_utils import NpEncoder

In [2]:
dataset_path = "../../dataset"
index_field = "timestamp"
data_field = "num_request"

BATCH_SIZE = 1
N_HISTORY = 32
N_LOOKBACK = 4
N_PREDICT = 2
LR = 1e-2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RESULT_ROOT_PATH="results"
# MODEL_NAME="lstm"
# MODEL_NAME="nlinear"
# MODEL_NAME="segrnn"
MODEL_NAME="patchtst"

TRAIN_EPOCH=20
EARLY_STOP_GAIN=0.01
EARLY_STOP_LOSS=0.02

start_index = -1
start_index_end = -1
start_index_lock = threading.Lock()

N_THREAD = 8

train_gt_scaled = {}
train_pd_scaled = {}
test_gt_scaled = {}
test_pd_scaled = {}

train_gt_original = {}
train_pd_original = {}
test_gt_original = {}
test_pd_original = {}

In [3]:
def get_data_file_list(dataset_path: str) -> List[str]:
    return os.listdir(dataset_path)


In [4]:
def read_dataset(csv_path: str) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(csv_path)
    return df[index_field].to_numpy(), df[data_field].to_numpy()

In [5]:
def learn_on_history(history_data: np.ndarray) -> Tuple[Evaluator, MinMaxScaler, Evaluator, MinMaxScaler]:
    train_set, test_set = split_data(history_data, N_LOOKBACK, N_PREDICT)
    train_set, train_scaler = min_max_normalization(train_set)
    test_set, test_scaler = min_max_normalization(test_set)
    train_dataset = TimeSeriesDataset(train_set, N_LOOKBACK, N_PREDICT)
    test_dataset = TimeSeriesDataset(test_set, N_LOOKBACK, N_PREDICT)
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    # model = LSTMMdel(1, 64, N_PREDICT, 2).to(DEVICE)
    # model=NLinear(N_LOOKBACK,N_PREDICT).to(DEVICE)
    # model = SegRNN(seq_len=N_LOOKBACK, pred_len=N_PREDICT, enc_in=1, d_model=64, dropout=0.5, rnn_type="lstm", dec_way="rmf", seg_len=1, channel_id=False, revin=False).to(DEVICE)
    model = PatchTST(configs=Config(enc_in=1, seq_len=N_LOOKBACK, pred_len=N_PREDICT, e_layers=1, n_heads=4, d_model=16, d_ff=16, dropout=0.5, fc_dropout=0.5, head_dropout=0.5, individual=False, patch_len=1, stride=1, padding_patch=False, revin=False, affine=False, subtract_last=False, decomposition=False, kernel_size=1)).to(DEVICE)
    
    # for name, param in model.named_parameters():
    #     if 'weight' in name:
    #         init.xavier_normal_(param)
    #     elif 'bias' in name:
    #         init.constant_(param, 0.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    trainer = Trainer(model, train_dataloader, loss_fn, optimizer, num_epochs=TRAIN_EPOCH, early_stop_gain=EARLY_STOP_GAIN, early_stop_loss=EARLY_STOP_LOSS, lr_scheduler=None, device=DEVICE)
    trainer.train_by_epoch()
    train_evaluator = Evaluator(model, train_dataloader, loss_fn, DEVICE)
    train_evaluator.evaluate()
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_evaluator = Evaluator(model, test_dataloader, loss_fn, DEVICE)
    test_evaluator.evaluate()
    return train_evaluator, train_scaler, test_evaluator, test_scaler

In [6]:
def save_to_dict_scaled(train_evaluator: Evaluator, test_evaluator: Evaluator, local_start_index: int):
    train_gt_scaled[local_start_index] = train_evaluator.get_gt()
    train_pd_scaled[local_start_index] = train_evaluator.get_pd()
    test_gt_scaled[local_start_index+N_HISTORY-N_LOOKBACK] = test_evaluator.get_gt()
    test_pd_scaled[local_start_index+N_HISTORY-N_LOOKBACK] = test_evaluator.get_pd()

In [7]:
def save_to_dict_original(train_evaluator: Evaluator, train_scaler: MinMaxScaler, test_evaluator: Evaluator, test_scaler: MinMaxScaler, local_start_index: int):
    tmp_train_gt_scaled, tmp_train_pd_scaled, tmp_test_gt_scaled, tmp_test_pd_scaled = train_evaluator.get_gt(), train_evaluator.get_pd(), test_evaluator.get_gt(), test_evaluator.get_pd()

    tmp_train_gt_original = np.hstack([train_scaler.inverse_transform(tmp_train_gt_scaled[:, i_dim]) for i_dim in range(tmp_train_gt_scaled.shape[1])])
    tmp_train_gt_original = np.expand_dims(tmp_train_gt_original, -1)
    tmp_train_pd_original = np.hstack([train_scaler.inverse_transform(tmp_train_pd_scaled[:, i_dim]) for i_dim in range(tmp_train_pd_scaled.shape[1])])
    tmp_train_pd_original = np.expand_dims(tmp_train_pd_original, -1)
    tmp_test_gt_original = np.hstack([test_scaler.inverse_transform(tmp_test_gt_scaled[:, i_dim]) for i_dim in range(tmp_test_gt_scaled.shape[1])])
    tmp_test_gt_original = np.expand_dims(tmp_test_gt_original, -1)
    tmp_test_pd_original = np.hstack([test_scaler.inverse_transform(tmp_test_pd_scaled[:, i_dim]) for i_dim in range(tmp_test_pd_scaled.shape[1])])
    tmp_test_pd_original = np.expand_dims(tmp_test_pd_original, -1)

    train_gt_original[local_start_index] = tmp_train_gt_original
    train_pd_original[local_start_index] = tmp_train_pd_original
    test_gt_original[local_start_index+N_HISTORY-N_LOOKBACK] = tmp_test_gt_original
    test_pd_original[local_start_index+N_HISTORY-N_LOOKBACK] = tmp_test_pd_original

In [8]:
def run_learning(np_data: np.ndarray):
    global start_index, start_index_end
    while start_index < start_index_end:
        start_index_lock.acquire()
        if start_index > start_index_end:
            start_index_lock.release()
        else:
            start_index += 1
            local_start_index = start_index
            start_index_lock.release()
            print(str(threading.current_thread().name) + "\t" + str(local_start_index))
            history_data = np_data[local_start_index:local_start_index+N_HISTORY]
            train_evaluator, train_scaler, test_evaluator, test_scaler = learn_on_history(history_data)
            save_to_dict_scaled(train_evaluator, test_evaluator, local_start_index)
            save_to_dict_original(train_evaluator, train_scaler, test_evaluator, test_scaler, local_start_index)

In [9]:
def save_to_file(result_dict: Dict[int, np.ndarray], file_name: str):
    save_dir = os.path.join(RESULT_ROOT_PATH, MODEL_NAME)
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    result_list=[result_dict[key] for key in sorted(result_dict.keys())]
    np_result=np.array(result_list)
    with open(os.path.join(save_dir, file_name), "w") as f:
        json.dump(np_result, f, indent=4, cls=NpEncoder)

In [10]:
data_file_list = get_data_file_list(dataset_path)
for file_name in data_file_list:
    
    np_index, np_data = read_dataset(os.path.join(dataset_path, file_name))
    np_data = np_data.reshape((-1, 1))
    
    start_index = -1
    start_index_end = len(np_data)-N_HISTORY
    
    train_gt_scaled = {}
    train_pd_scaled = {}
    test_gt_scaled = {}
    test_pd_scaled = {}

    train_gt_original = {}
    train_pd_original = {}
    test_gt_original = {}
    test_pd_original = {}
    
    thread_list = []
    for _ in range(N_THREAD):
        thread_list.append(threading.Thread(target=run_learning, args=[np_data]))
    for t in thread_list:
        t.start()
    for t in thread_list:
        t.join()

    save_to_file(train_gt_scaled, file_name.split(".")[0]+"_gt_train_scaled.json")
    save_to_file(train_pd_scaled, file_name.split(".")[0]+"_pd_train_scaled.json")
    save_to_file(test_gt_scaled, file_name.split(".")[0]+"_gt_test_scaled.json")
    save_to_file(test_pd_scaled, file_name.split(".")[0]+"_pd_test_scaled.json")
    
    save_to_file(train_gt_original, file_name.split(".")[0]+"_gt_train_original.json")
    save_to_file(train_pd_original, file_name.split(".")[0]+"_pd_train_original.json")
    save_to_file(test_gt_original, file_name.split(".")[0]+"_gt_test_original.json")
    save_to_file(test_pd_original, file_name.split(".")[0]+"_pd_test_original.json")

    

    

Thread-4	0
Thread-5	1
Thread-6	2
Thread-7	3
Thread-8	4
Thread-9	5
Thread-10	6
Thread-11	7
Thread-5	8
Thread-4	9
Thread-8	10
Thread-5	11
Thread-4	12
Thread-5	13
Thread-8	14
Thread-11	15
Thread-4	16
Thread-5	17
Thread-4	18
Thread-8	19
Thread-9	20
Thread-5	21
Thread-11	22
Thread-4	23
Thread-5	24
Thread-8	25
Thread-4	26
Thread-5	27
Thread-4	28
Thread-8	29
Thread-11	30
Thread-5	31
Thread-9	32
Thread-4	33
Thread-5	34
Thread-8	35
Thread-5	36
Thread-4	37
Thread-11	38
Thread-6	39
Thread-7	40
Thread-10	41
Thread-5	42
Thread-8	43
Thread-4	44
Thread-5	45
Thread-4	46
Thread-9	47
Thread-8	48
Thread-11	49
Thread-5	50
Thread-4	51
Thread-5	52
Thread-8	53
Thread-4	54
Thread-5	55
Thread-11	56
Thread-4	57
Thread-8	58
Thread-5	59
Thread-9	60
Thread-4	61
Thread-5	62
Thread-8	63
Thread-11	64
Thread-4	65
Thread-5	66
Thread-4	67
Thread-5	68
Thread-8	69
Thread-4	70
Thread-5	71
Thread-11	72
Thread-8	73
Thread-9	74
Thread-5	75
Thread-4	76
Thread-5	77
Thread-6	78
Thread-4	79
Thread-8	80
Thread-5	81
Thread-11	82
Th